# ECSS Compliance Demo: ESSB-ST-U-007 Space Debris Mitigation

End-to-end verification that factpy can encode and audit ECSS compliance rules.
Uses existing SDK/Service API — no core code changes.

**Scenario A:** `SENTINEL-7` (`single`)
- Disposal success probability: 92% (threshold: 90%)
- Collision probability: 0.05% (threshold: 0.1%)
- Passivation: `complete`

**Scenario B:** `SWARM-9` (`constellation`)
- Disposal success probability: 92% (threshold: 95%)
- Collision probability: 0.12% (threshold: 0.1%)
- Passivation: `incomplete`

**Validation slices in this notebook:**
1. Base ECSS checks: disposal / collision / passivation
2. Top-level compliance = conjunction of sub-checks via `ruleref` chain
3. Explicit non-compliance derivation for the constellation mission
4. Profile-dependent branching: `single` vs `constellation`
5. Souffle provenance on positive and negative flat rules, plus the current composed-rule boundary

**Full pipeline demonstrated:**
Schema → Rules → Registry → Facts → Evaluate → Evidence Tree → Certainty
→ Narrative → NL → Top-Level Composition → Non-Compliance → Souffle Provenance → Audit → Static HTML


## 1. Schema: Mission entity with ECSS predicates

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path
from pprint import pprint

# Ensure src/ is on the Python path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from factpy_kernel.sdk import (
    SDKStore,
    Entity,
    Identity,
    Field,
    Rule,
    Pred,
    vars as sdk_vars,
)
from factpy_kernel.sdk.dsl.rule import RuleRef
from factpy_kernel.authoring import FileAuthoringRegistry
from factpy_kernel.domains.ecss import (
    ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
    ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID,
    ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
    ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID,
    ECSS_COMPLIANCE_STATUS_PRED_ID,
    ECSS_REQUIREMENT_PRED_ID,
    ECSS_VERIFICATION_METHOD_PRED_ID,
)
from factpy_kernel.service.runtime_v1 import (
    open_runtime_session,
    close_runtime_session,
    reset_runtime_sessions_for_tests,
    write_runtime_fact,
    evaluate_runtime_derivation,
    explain_runtime_summary,
    explain_runtime_narrative,
    explain_runtime_nl,
    export_runtime_package,
    accept_runtime_derivation,
)
from factpy_kernel.audit import AuditQuery, load_audit_package


In [ ]:
class Mission(Entity):
    mission_id: str = Identity(primary_key=True)
    locale: str = Identity()
    name: str = Field(cardinality="single")
    orbit_type: str = Field(cardinality="single")
    mission_profile: str = Field(cardinality="single")
    passivation_status: str = Field(cardinality="single")
    overall_compliance_status: str = Field(cardinality="single")


class Ecss(Entity):
    """Schema-only ECSS predicate anchor used to declare `ecss:*` predicates."""

    anchor_id: str = Identity(primary_key=True)
    collision_probability_ppm: int = Field(cardinality="single")
    collision_probability_threshold_ppm: int = Field(cardinality="single")
    disposal_success_probability_ppm: int = Field(cardinality="single")
    disposal_success_threshold_ppm: int = Field(cardinality="single")
    requirement: str = Field(cardinality="single")
    verification_method: str = Field(cardinality="multi")
    compliance_status: str = Field(cardinality="single")
    requirement_rid: str = Field(cardinality="multi")
    review_milestone: str = Field(cardinality="single")


sdk = SDKStore([Mission, Ecss])

print("=== Schema ready ===")
ecss_preds = [p["pred_id"] for p in sdk.schema_ir["predicates"] if p["pred_id"].startswith("ecss:")]
print(f"  ECSS predicates registered: {ecss_preds}")


## 2. Rules: ESSB-ST-U-007 compliance checks

In [3]:
# Rule 1: Disposal probability check
# "disposal success probability >= threshold"
with sdk_vars("m", "prob", "threshold") as (m, prob, threshold):
    disposal_check_rule = Rule(
        id="q.essb_u007_disposal_check",
        version="1.0.0",
        select=[m, prob],
        where=[
            Pred(ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID, m, prob),
            Pred(ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID, m, threshold),
            prob >= threshold,
        ],
        expose=True,
        condition_weights={
            "b0.a0": 0.8,
            "b0.a1": 0.5,
        },
    )

# Rule 2: Collision probability check
# "collision probability <= threshold"
with sdk_vars("m", "prob", "threshold") as (m, prob, threshold):
    collision_check_rule = Rule(
        id="q.essb_u007_collision_check",
        version="1.0.0",
        select=[m, prob],
        where=[
            Pred(ECSS_COLLISION_PROBABILITY_PPM_PRED_ID, m, prob),
            Pred(ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID, m, threshold),
            threshold >= prob,
        ],
        expose=True,
        condition_weights={
            "b0.a0": 0.9,
            "b0.a1": 0.4,
        },
    )

# Rule 3: Passivation check
# "mission passivation status is 'complete'"
with sdk_vars("m", "status") as (m, status):
    passivation_check_rule = Rule(
        id="q.essb_u007_passivation_check",
        version="1.0.0",
        select=[m, status],
        where=[
            Pred("mission:passivation_status", m, status),
            status == "complete",
        ],
        expose=True,
        condition_weights={
            "b0.a0": 1.0,
        },
    )

# Rule 4: Single-mission top-level compliance branch
with sdk_vars("m", "profile", "status", "prob", "collision_prob", "pass_status") as (
    m,
    profile,
    status,
    prob,
    collision_prob,
    pass_status,
):
    single_branch_compliance_rule = Rule(
        id="q.essb_u007_single_branch_compliance",
        version="1.0.0",
        select=[m, status],
        where=[
            Pred("mission:mission_profile", m, profile),
            profile == "single",
            RuleRef(disposal_check_rule)(m, prob),
            RuleRef(collision_check_rule)(m, collision_prob),
            RuleRef(passivation_check_rule)(m, pass_status),
            status == "compliant",
        ],
        expose=True,
        condition_weights={
            "b0.a0": 0.2,
            "b0.a1": 0.5,
            "b0.a2": 0.4,
            "b0.a3": 0.8,
        },
    )

# Rule 5: Constellation top-level compliance branch
with sdk_vars("m", "profile", "status", "prob", "collision_prob", "pass_status") as (
    m,
    profile,
    status,
    prob,
    collision_prob,
    pass_status,
):
    constellation_branch_compliance_rule = Rule(
        id="q.essb_u007_constellation_branch_compliance",
        version="1.0.0",
        select=[m, status],
        where=[
            Pred("mission:mission_profile", m, profile),
            profile == "constellation",
            RuleRef(disposal_check_rule)(m, prob),
            RuleRef(collision_check_rule)(m, collision_prob),
            RuleRef(passivation_check_rule)(m, pass_status),
            status == "compliant",
        ],
        expose=True,
        condition_weights={
            "b0.a0": 0.2,
            "b0.a1": 0.5,
            "b0.a2": 0.4,
            "b0.a3": 0.8,
        },
    )

# Rule 6: Overall compliance = choose the matching branch
with sdk_vars("m", "status") as (m, status):
    overall_compliance_rule = Rule(
        id="q.essb_u007_overall_compliance",
        version="1.0.0",
        select=[m, status],
        where=[
            [RuleRef(single_branch_compliance_rule)(m, status)],
            [RuleRef(constellation_branch_compliance_rule)(m, status)],
        ],
        expose=True,
    )

# Rule 7: Explicit disposal non-compliance for constellation missions
with sdk_vars("m", "profile", "prob", "threshold", "status") as (m, profile, prob, threshold, status):
    constellation_disposal_noncompliance_rule = Rule(
        id="q.essb_u007_constellation_disposal_noncompliance",
        version="1.0.0",
        select=[m, status],
        where=[
            Pred("mission:mission_profile", m, profile),
            profile == "constellation",
            Pred(ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID, m, prob),
            Pred(ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID, m, threshold),
            threshold > prob,
            status == "non_compliant",
        ],
        expose=True,
    )

# Rule 8: Explicit passivation non-compliance
with sdk_vars("m", "pass_status", "status") as (m, pass_status, status):
    passivation_noncompliance_rule = Rule(
        id="q.essb_u007_passivation_noncompliance",
        version="1.0.0",
        select=[m, status],
        where=[
            Pred("mission:passivation_status", m, pass_status),
            pass_status == "incomplete",
            status == "non_compliant",
        ],
        expose=True,
    )

# Rule 9: Overall non-compliance = any explicit failure branch
with sdk_vars("m", "status") as (m, status):
    overall_noncompliance_rule = Rule(
        id="q.essb_u007_overall_noncompliance",
        version="1.0.0",
        select=[m, status],
        where=[
            [RuleRef(constellation_disposal_noncompliance_rule)(m, status)],
            [RuleRef(passivation_noncompliance_rule)(m, status)],
        ],
        expose=True,
    )

all_rules = [
    disposal_check_rule,
    collision_check_rule,
    passivation_check_rule,
    single_branch_compliance_rule,
    constellation_branch_compliance_rule,
    overall_compliance_rule,
    constellation_disposal_noncompliance_rule,
    passivation_noncompliance_rule,
    overall_noncompliance_rule,
]

print("=== Rules defined ===")
for rule in all_rules:
    print(f"  {rule.id}")


=== Rules defined ===
  q.essb_u007_disposal_check
  q.essb_u007_collision_check
  q.essb_u007_passivation_check
  q.essb_u007_single_branch_compliance
  q.essb_u007_constellation_branch_compliance
  q.essb_u007_overall_compliance
  q.essb_u007_constellation_disposal_noncompliance
  q.essb_u007_passivation_noncompliance
  q.essb_u007_overall_noncompliance


## 3. Registry: register rules

In [4]:
registry_dir = tempfile.mkdtemp(prefix="ecss_demo_")
registry = FileAuthoringRegistry(Path(registry_dir))
registry.upsert_schema_ir(sdk.schema_ir)
for rule in all_rules:
    registry.register_rule_spec(sdk._compile_rule_input(rule))

print(f"=== Registry ready: {registry_dir} ===")
print(f"  Rules registered: {len(all_rules)}")


=== Registry ready: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/ecss_demo_8jnb_zr1 ===
  Rules registered: 9


## 4. Data: two missions (`single` + `constellation`)

In [5]:
with sdk.batch() as tx:
    sentinel = tx.entity(Mission, mission_id="SENTINEL-7", locale="en")
    sentinel.name.set("Sentinel-7 LEO Observatory")
    sentinel.orbit_type.set("LEO")
    sentinel.mission_profile.set("single")
    sentinel.passivation_status.set("complete")
    swarm = tx.entity(Mission, mission_id="SWARM-9", locale="en")
    swarm.name.set("Swarm-9 Constellation Vehicle")
    swarm.orbit_type.set("LEO")
    swarm.mission_profile.set("constellation")
    swarm.passivation_status.set("incomplete")
    tx.commit()

sentinel_ref = sdk.ref(Mission, mission_id="SENTINEL-7", locale="en")
swarm_ref = sdk.ref(Mission, mission_id="SWARM-9", locale="en")
mission_ref = sentinel_ref  # legacy alias used by the single-mission base checks below

print("=== Mission entities created ===")
print(f"  SENTINEL-7: {sentinel_ref}")
print(f"  SWARM-9:    {swarm_ref}")


=== Mission entities created ===
  SENTINEL-7: idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq
  SWARM-9:    idref_v1:Mission:3m6gyqnpupjytrtuqamjkkqcmonf5tvubqgeto377iimicvpcjka


## 5. Runtime: write facts + evaluate

In [6]:
reset_runtime_sessions_for_tests()
session_resp = open_runtime_session({"registry_root": registry_dir})
session_id = session_resp["session"]["session_id"]

# SENTINEL-7: compliant single mission
write_runtime_fact(session_id, {
    "pred_id": "mission:mission_profile",
    "e_ref": sentinel_ref,
    "rest_terms": [["string", "single"]],
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": "mission:passivation_status",
    "e_ref": sentinel_ref,
    "rest_terms": [["string", "complete"]],
    "meta": {"confidence": 0.99},
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
    "e_ref": sentinel_ref,
    "rest_terms": [["int", 920000]],
    "meta": {"confidence": 0.85},
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID,
    "e_ref": sentinel_ref,
    "rest_terms": [["int", 900000]],
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
    "e_ref": sentinel_ref,
    "rest_terms": [["int", 500]],
    "meta": {"confidence": 0.70},
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID,
    "e_ref": sentinel_ref,
    "rest_terms": [["int", 1000]],
}, kind="add")

# SWARM-9: non-compliant constellation mission
write_runtime_fact(session_id, {
    "pred_id": "mission:mission_profile",
    "e_ref": swarm_ref,
    "rest_terms": [["string", "constellation"]],
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": "mission:passivation_status",
    "e_ref": swarm_ref,
    "rest_terms": [["string", "incomplete"]],
    "meta": {"confidence": 0.95},
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
    "e_ref": swarm_ref,
    "rest_terms": [["int", 920000]],
    "meta": {"confidence": 0.82},
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_DISPOSAL_SUCCESS_THRESHOLD_PPM_PRED_ID,
    "e_ref": swarm_ref,
    "rest_terms": [["int", 950000]],
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
    "e_ref": swarm_ref,
    "rest_terms": [["int", 1200]],
    "meta": {"confidence": 0.72},
}, kind="add")
write_runtime_fact(session_id, {
    "pred_id": ECSS_COLLISION_PROBABILITY_THRESHOLD_PPM_PRED_ID,
    "e_ref": swarm_ref,
    "rest_terms": [["int", 1000]],
}, kind="add")

print("=== Runtime facts written ===")
print("  SENTINEL-7  profile=single        disposal=920000/900000  collision=500/1000   passivation=complete")
print("  SWARM-9     profile=constellation disposal=920000/950000  collision=1200/1000  passivation=incomplete")


=== Runtime facts written ===
  SENTINEL-7  profile=single        disposal=920000/900000  collision=500/1000   passivation=complete
  SWARM-9     profile=constellation disposal=920000/950000  collision=1200/1000  passivation=incomplete


## 6. Evaluate: Disposal Success Probability Check

In [7]:
eval_disposal = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.disposal_check",
        "version": "1.0.0",
        "target": ECSS_DISPOSAL_SUCCESS_PROBABILITY_PPM_PRED_ID,
        "head_vars": ["$m", "$prob"],
        "where": [
            ["ruleref", "q.essb_u007_disposal_check", "1.0.0", ["$m", "$prob"]],
        ],
        "mode": "native",
    }
})

if eval_disposal["ok"] and eval_disposal["evaluation"]["candidates"]:
    disposal_cand = eval_disposal["evaluation"]["candidates"][0]
    disposal_cid = disposal_cand["candidate_id"]
    print(f"  ✅ Disposal check PASSED")
    print(f"  candidate_id: {disposal_cid}")
    print(f"  confidence_kind: {disposal_cand['confidence_kind']}")

    # Summary
    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": disposal_cid})
    if summary["ok"]:
        print(f"\n  --- Evidence Tree Summary ---")
        s = summary["summary"]
        print(f"  support_kind: {s.get('support_kind')}")
        print(f"  witness_assertion_count: {s.get('witness_assertion_count')}")
        print(f"  rule_ref_count: {s.get('rule_ref_count')}")
        cs = summary.get("certainty_summary")
        if cs:
            print(f"\n  --- Certainty Summary ---")
            print(f"  aggregate_certainty: {cs['aggregate_certainty']}")
            print(f"  aggregation: {cs['aggregation']}")
            for c in cs["conditions"]:
                print(f"    {c['atom_key']}: weight={c['weight']}, impact={c['impact']}")

    # Narrative
    narrative = explain_runtime_narrative(session_id, {"kind": "candidate", "id": disposal_cid})
    if narrative["ok"]:
        narr = narrative["narrative"]
        print(f"\n  --- Narrative ---")
        for key in ["headline", "overview_lines", "evidence_lines", "rule_chain_lines", "certainty_lines"]:
            val = narr.get(key)
            if val:
                if isinstance(val, list):
                    for line in val:
                        print(f"  [{key}] {line}")
                else:
                    print(f"  [{key}] {val}")
else:
    print(f"  ❌ Disposal check FAILED or no candidates")
    print(f"  Response: {eval_disposal}")

  ✅ Disposal check PASSED
  candidate_id: cand_v2:8cb762fcea32850c2d7c0e51682ec703eb6e10f059d22836bea61946c71abffb
  confidence_kind: certainty

  --- Evidence Tree Summary ---
  support_kind: native_binding_v1
  witness_assertion_count: 2
  rule_ref_count: 1

  --- Certainty Summary ---
  aggregate_certainty: 0.5
  aggregation: bottleneck
    b0.a0: weight=0.8, impact=0.68
    b0.a1: weight=0.5, impact=0.5
    b0.a2: weight=None, impact=None

  --- Narrative ---
  [headline] Candidate cand_v2:8cb762fcea32850c2d7c0e51682ec703eb6e10f059d22836bea61946c71abffb uses support kind native_binding_v1 across 12 tree node(s).
  [overview_lines] Root result kind: fact.
  [overview_lines] Role counts: structural=4, witness=4, constraint=2, rule_chain=2, terminal=0, degraded=0.
  [overview_lines] Recursive depth: 1.
  [evidence_lines] Witness assertions: 2.
  [evidence_lines] Witness nodes: 4; constraint nodes: 2.
  [rule_chain_lines] Rule reference nodes: 1.
  [rule_chain_lines] Recursive proof de

## 7. Evaluate: Collision Probability Check

In [8]:
eval_collision = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.collision_check",
        "version": "1.0.0",
        "target": ECSS_COLLISION_PROBABILITY_PPM_PRED_ID,
        "head_vars": ["$m", "$prob"],
        "where": [
            ["ruleref", "q.essb_u007_collision_check", "1.0.0", ["$m", "$prob"]],
        ],
        "mode": "native",
    }
})

if eval_collision["ok"] and eval_collision["evaluation"]["candidates"]:
    collision_cand = eval_collision["evaluation"]["candidates"][0]
    collision_cid = collision_cand["candidate_id"]
    print(f"  ✅ Collision check PASSED")
    print(f"  candidate_id: {collision_cid}")
    print(f"  confidence_kind: {collision_cand['confidence_kind']}")

    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": collision_cid})
    if summary["ok"]:
        cs = summary.get("certainty_summary")
        if cs:
            print(f"\n  --- Certainty Summary ---")
            print(f"  aggregate_certainty: {cs['aggregate_certainty']}")
            for c in cs["conditions"]:
                print(f"    {c['atom_key']}: weight={c['weight']}, impact={c['impact']}")

    nl = explain_runtime_nl(session_id, {"kind": "candidate", "id": collision_cid})
    if nl["ok"]:
        print(f"\n  --- NL Explanation ---")
        for i, p in enumerate(nl["explain_nl"]["paragraphs"], 1):
            print(f"  [{i}] {p[:150]}...")
else:
    print(f"  ❌ Collision check FAILED or no candidates")
    print(f"  Response: {eval_collision}")

  ✅ Collision check PASSED
  candidate_id: cand_v2:0bcde3249ed9fcc78370dd5577f318eba66536ec855a79a72b94dda98768a221
  confidence_kind: certainty

  --- Certainty Summary ---
  aggregate_certainty: 0.4
    b0.a0: weight=0.9, impact=0.63
    b0.a1: weight=0.4, impact=0.4
    b0.a2: weight=None, impact=None

  --- NL Explanation ---
  [1] Candidate cand_v2:0bcde3249ed9fcc78370dd5577f318eba66536ec855a79a72b94dda98768a221 uses support kind native_binding_v1 across 12 tree node(s). Root re...
  [2] Evidence summary: Witness assertions: 2. Witness nodes: 4; constraint nodes: 2....
  [3] Rule-chain summary: Rule reference nodes: 1. Recursive proof depth: 1....
  [4] Terminal and drill-down summary: No unresolved support or recursion boundaries were encountered. Open referenced support branches to inspect recursive...
  [5] Certainty summary: Certainty (eligible child-proof subtree): aggregate certainty (bottleneck): 0.4. Condition b0.a1 (predicate_witness_group): weight=...


## 8. Evaluate: Passivation Check

In [9]:
eval_passivation = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.passivation_check",
        "version": "1.0.0",
        "target": "mission:passivation_status",
        "head_vars": ["$m", "$status"],
        "where": [
            ["ruleref", "q.essb_u007_passivation_check", "1.0.0", ["$m", "$status"]],
        ],
        "mode": "native",
    }
})

if eval_passivation["ok"] and eval_passivation["evaluation"]["candidates"]:
    pass_cand = eval_passivation["evaluation"]["candidates"][0]
    pass_cid = pass_cand["candidate_id"]
    print(f"  ✅ Passivation check PASSED")
    print(f"  confidence_kind: {pass_cand['confidence_kind']}")

    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": pass_cid})
    if summary["ok"]:
        cs = summary.get("certainty_summary")
        if cs:
            print(f"  aggregate_certainty: {cs['aggregate_certainty']}")
            for c in cs["conditions"]:
                if c["weight"] is not None:
                    print(f"    {c['atom_key']}: weight={c['weight']}, impact={c['impact']}")
else:
    print(f"  ❌ Passivation check FAILED")
    print(f"  Response: {eval_passivation}")


  ✅ Passivation check PASSED
  confidence_kind: certainty
  aggregate_certainty: 0.99
    b0.a0: weight=1.0, impact=0.99


## 9. Evaluate: Top-Level Compliance (`ruleref` chain)

In [10]:
eval_overall_compliance = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.overall_compliance",
        "version": "1.0.0",
        "target": "mission:overall_compliance_status",
        "head_vars": ["$m", "$status"],
        "where": [
            ["ruleref", "q.essb_u007_overall_compliance", "1.0.0", ["$m", "$status"]],
        ],
        "mode": "native",
    }
})

if eval_overall_compliance["ok"] and eval_overall_compliance["evaluation"]["candidates"]:
    overall_cand = eval_overall_compliance["evaluation"]["candidates"][0]
    overall_cid = overall_cand["candidate_id"]
    terms = overall_cand["payload"]["terms"]
    print("  ✅ Top-level compliance PASSED")
    print(f"  mission: {terms[0]['value']}")
    print(f"  status:  {terms[1]['value']}")
    print(f"  candidate_id: {overall_cid}")

    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": overall_cid})
    if summary["ok"]:
        s = summary["summary"]
        print("\n  --- Summary ---")
        print(f"  support_kind: {s.get('support_kind')}")
        print(f"  rule_ref_count: {s.get('rule_ref_count')}")
        print(f"  witness_assertion_count: {s.get('witness_assertion_count')}")
else:
    print("  ❌ Top-level compliance FAILED")
    print(f"  Response: {eval_overall_compliance}")


  ✅ Top-level compliance PASSED
  mission: idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq
  status:  compliant
  candidate_id: cand_v2:3176226cb64eb9daca17f7a13dbc8e99d11f8cc8246309d1879aca566443752b

  --- Summary ---
  support_kind: native_binding_v1
  rule_ref_count: 5
  witness_assertion_count: 6


## 10. Evaluate: Explicit Non-Compliant Case

In [11]:
eval_overall_noncompliance = evaluate_runtime_derivation(session_id, {
    "derivation": {
        "derivation_id": "drv.overall_noncompliance",
        "version": "1.0.0",
        "target": "mission:overall_compliance_status",
        "head_vars": ["$m", "$status"],
        "where": [
            ["ruleref", "q.essb_u007_overall_noncompliance", "1.0.0", ["$m", "$status"]],
        ],
        "mode": "native",
    }
})

if eval_overall_noncompliance["ok"] and eval_overall_noncompliance["evaluation"]["candidates"]:
    noncompliance_cand = eval_overall_noncompliance["evaluation"]["candidates"][0]
    noncompliance_cid = noncompliance_cand["candidate_id"]
    terms = noncompliance_cand["payload"]["terms"]
    print("  ✅ Explicit non-compliance derivation PASSED")
    print(f"  mission: {terms[0]['value']}")
    print(f"  status:  {terms[1]['value']}")
    print(f"  candidate_id: {noncompliance_cid}")

    summary = explain_runtime_summary(session_id, {"kind": "candidate", "id": noncompliance_cid})
    if summary["ok"]:
        s = summary["summary"]
        print("\n  --- Summary ---")
        print(f"  support_kind: {s.get('support_kind')}")
        print(f"  rule_ref_count: {s.get('rule_ref_count')}")
        print(f"  witness_assertion_count: {s.get('witness_assertion_count')}")
else:
    print("  ❌ Explicit non-compliance derivation FAILED")
    print(f"  Response: {eval_overall_noncompliance}")


  ✅ Explicit non-compliance derivation PASSED
  mission: idref_v1:Mission:3m6gyqnpupjytrtuqamjkkqcmonf5tvubqgeto377iimicvpcjka
  status:  non_compliant
  candidate_id: cand_v2:08bcc69fcc393e587ba3898eb1cfcc1adead45981331a8ac414573926973e6d1

  --- Summary ---
  support_kind: native_binding_v1
  rule_ref_count: 2
  witness_assertion_count: 3


## 11. Souffle Provenance: Flat Rules And Composed Rule Chain


In [12]:
from factpy_kernel.adapters.souffle.provenance import run_package_provenance
from factpy_kernel.adapters.souffle.runner import run_package


def render_proof_tree(title, rule, query_rel):
    compiled_where = sdk._compile_rule_input(rule)["where"]
    provenance_pkg_dir = tempfile.mkdtemp(prefix=f"ecss_{query_rel}_")
    prov_export = export_runtime_package(session_id, {
        "out_dir": provenance_pkg_dir,
        "package_kind": "inference",
        "query": {
            "where": compiled_where,
            "query_rel": query_rel,
            "registry_root": registry_dir,
        },
    })
    assert prov_export["ok"], f"provenance export failed: {prov_export}"

    run_package(Path(provenance_pkg_dir), ["__query__"], engine="souffle")
    out_path = Path(provenance_pkg_dir) / "outputs" / f"{query_rel}.out.facts"
    rows = [line.split("\t") for line in out_path.read_text().splitlines() if line.strip()]
    assert rows, f"no rows returned for {query_rel}"

    provenance_query = query_rel + "(" + ", ".join(f'\"{cell}\"' for cell in rows[0]) + ")"
    proof_trees = run_package_provenance(Path(provenance_pkg_dir), [provenance_query])
    assert proof_trees, f"no proof trees returned for {query_rel}"

    tree = proof_trees[0]
    print(f"\n{'=' * 72}")
    print(title)
    print(f"Query: {tree.query}")
    print(f"{'=' * 72}")

    def print_proof_node(node, indent=0):
        prefix = "  " * indent
        args_display = ", ".join(node.args[:3])
        if len(node.args) > 3:
            args_display += ", ..."
        label = f"{node.relation}({args_display})"
        if node.node_type == "axiom":
            print(f"{prefix}FACT: {label}")
        elif node.node_type == "negation":
            print(f"{prefix}NOT: {label}")
        elif node.node_type == "subproof":
            print(f"{prefix}SUBPROOF: {label}  (depth limit)")
        else:
            rule = node.rule_number or "?"
            print(f"{prefix}RULE {rule}: {label}")
        for child in node.children:
            print_proof_node(child, indent + 1)

    print_proof_node(tree.root)
    return tree


flat_positive_tree = render_proof_tree(
    title="SOUFFLE PROOF TREE: Disposal Success Probability Check (positive case)",
    rule=disposal_check_rule,
    query_rel="disposal_check_query",
)

flat_negative_tree = render_proof_tree(
    title="SOUFFLE PROOF TREE: Constellation Disposal Non-Compliance (negative case)",
    rule=constellation_disposal_noncompliance_rule,
    query_rel="constellation_disposal_noncompliance_query",
)

composed_tree = render_proof_tree(
    title="SOUFFLE PROOF TREE: Overall Compliance (composed ruleref chain)",
    rule=overall_compliance_rule,
    query_rel="overall_compliance_query",
)

print(f"\n{'=' * 72}")
print("KEY OBSERVATIONS")
print("  * Positive flat-rule provenance shows assertion identity and comparison leaves.")
print("  * Negative flat-rule provenance shows failing-threshold derivation for the constellation mission.")
print("  * Composed provenance now traverses the top-level ECSS ruleref chain via query-bearing export.")
print("  * registry_root is only needed for composed query export; flat rules continue to work unchanged.")
print(f"{'=' * 72}")



SOUFFLE PROOF TREE: Disposal Success Probability Check (positive case)
Query: disposal_check_query("idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq", "920000", "900000")
RULE (R1): disposal_check_query(idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq, 920000, 900000)
  RULE (R1): p_ecss_disposal__success__probability__ppm(idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq, 920000)
    RULE (R1): chosen_asrt__p_ecss_disposal__success__probability__ppm(idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq, 54ea069b786a462296c0d593b274ef1a)
      SUBPROOF: cand__p_ecss_disposal__success__probability__ppm(0)  (depth limit)
      SUBPROOF: max_ts__p_ecss_disposal__success__probability__ppm(1)  (depth limit)
      NOT: better_asrt__p_ecss_disposal__success__probability__ppm(idref_v1:Mission:665jxtxmzu7lvmtmquths2kggo7terrm7fnwcqdouj6g2uyyelhq, 54ea069b786a462296c0d593b274ef1a)
    FACT: claim_arg(54ea069b786a46229

## 12. Audit: export + round-trip

In [13]:
# Accept all candidates (one at a time, singular "candidate" key)
all_evals = [
    ("disposal", eval_disposal),
    ("collision", eval_collision),
    ("passivation", eval_passivation),
    ("overall_compliance", eval_overall_compliance),
    ("overall_noncompliance", eval_overall_noncompliance),
]

for label, eval_resp in all_evals:
    if eval_resp["ok"] and eval_resp["evaluation"]["candidates"]:
        for cand in eval_resp["evaluation"]["candidates"]:
            resp = accept_runtime_derivation(session_id, {"candidate": cand})
            assert resp["ok"], f"accept {label} failed: {resp}"
            print(f"  ✅ Accepted {label} candidate")

# Export audit package
audit_dir = tempfile.mkdtemp(prefix="ecss_audit_")
export_resp = export_runtime_package(session_id, {
    "out_dir": audit_dir,
    "package_kind": "audit",
})
assert export_resp["ok"], f"export failed: {export_resp}"
print(f"\n  Export: ok={export_resp['ok']}")

# Load and query
pkg = load_audit_package(audit_dir)
aq = AuditQuery(pkg)

candidates = aq.list_candidates()
print(f"  Audit candidates: {len(candidates)}")

for cand_row in candidates:
    cid = cand_row["candidate_id"]

    tree = aq.get_candidate_evidence_tree(cid)
    if tree:
        print(f"\n  --- Audit Evidence Tree for {cid[:30]}... ---")
        print(f"  root node_kind: {tree.get('root', {}).get('node_kind', '?')}")
        root_children = tree.get('root', {}).get('children', [])
        for child in root_children:
            nk = child.get('node_kind', '?')
            if nk == 'rule_ref_section':
                for rr in child.get('children', []):
                    print(f"    rule_ref: {rr.get('rule_ref_id', '?')} v{rr.get('rule_ref_version', '?')}")

    cs = aq.get_candidate_certainty_summary(cid)
    if cs:
        print(f"  certainty: aggregate={cs['aggregate_certainty']}, aggregation={cs['aggregation']}")

    narr = aq.get_candidate_evidence_tree_narrative(cid)
    if narr:
        cert_lines = narr.get("certainty_lines", [])
        if cert_lines:
            print(f"  narrative certainty_lines: {len(cert_lines)} lines")

print(f"\n  Audit package: {audit_dir}")


  ✅ Accepted disposal candidate
  ✅ Accepted collision candidate
  ✅ Accepted passivation candidate
  ✅ Accepted overall_compliance candidate
  ✅ Accepted overall_noncompliance candidate

  Export: ok=True
  Audit candidates: 5

  --- Audit Evidence Tree for cand_v2:08bcc69fcc393e587ba389... ---
  root node_kind: candidate_result
    rule_ref: q.essb_u007_overall_noncompliance v1.0.0

  --- Audit Evidence Tree for cand_v2:0bcde3249ed9fcc78370dd... ---
  root node_kind: candidate_result
    rule_ref: q.essb_u007_collision_check v1.0.0
  certainty: aggregate=0.4, aggregation=bottleneck
  narrative certainty_lines: 4 lines

  --- Audit Evidence Tree for cand_v2:3176226cb64eb9daca17f7... ---
  root node_kind: candidate_result
    rule_ref: q.essb_u007_overall_compliance v1.0.0

  --- Audit Evidence Tree for cand_v2:8cb762fcea32850c2d7c0e... ---
  root node_kind: candidate_result
    rule_ref: q.essb_u007_disposal_check v1.0.0
  certainty: aggregate=0.5, aggregation=bottleneck
  narrative c

## 13. Static site

In [14]:
from factpy_kernel.audit.static_ui import render_audit_static_site

site_dir = tempfile.mkdtemp(prefix="ecss_site_")
render_audit_static_site(audit_dir, site_dir)

site_files = sorted(Path(site_dir).rglob("*.html"))
print(f"Static site: {len(site_files)} pages at {site_dir}")
for f in site_files[:10]:
    print(f"  {f.relative_to(site_dir)}")

Static site: 41 pages at /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/ecss_site_u9surr0d
  assertions/00167949a2204091bee18a5b1085d0e1.html
  assertions/1b163ce2ecbc4a0aa3b0e56a490d0c82.html
  assertions/1c0f8a45b5a74b45a404cf3300a4d022.html
  assertions/1c5ca8b60d68419a9e192d9ab1bba55a.html
  assertions/26dd87c7c51743868dfd3fffc3e1d5aa.html
  assertions/2f7ca95ace864e3ba49f02d31fdd1336.html
  assertions/43395b4d9ad748afac5ea854b890c706.html
  assertions/54ea069b786a462296c0d593b274ef1a.html
  assertions/58af107617044328a41e709e5ba8c949.html
  assertions/5df0fae7dcb6498a9095639e07e97f84.html


## 14. Cleanup

In [15]:
close_runtime_session(session_id)
reset_runtime_sessions_for_tests()

print("ECSS COMPLIANCE DEMO COMPLETE")
print(f"\nStatic audit site: {site_dir}")
print("Open candidate_evidence/*.html in a browser to see the evidence tree.")

ECSS COMPLIANCE DEMO COMPLETE

Static audit site: /var/folders/05/6btr2vg13b9gvgs3gxt8fw_40000gn/T/ecss_site_u9surr0d
Open candidate_evidence/*.html in a browser to see the evidence tree.
